# Web Scraping Exercise

Web Scraping allows you to gather large volumes of data from diverse and real-time online sources. This data can be crucial for enriching your datasets, filling in gaps, and providing current information that enhances the quality and relevance of your analysis. Web scraping enables you to collect data that might not be readily available through traditional APIs or databases, offering a competitive edge by incorporating unique and comprehensive insights. Moreover, it automates the data collection process, saving time and resources while ensuring a scalable approach to continuously updating and maintaining your datasets.

Ethical web scraping involves respecting website terms of service, avoiding overloading servers, and ensuring that the collected data is used responsibly and in compliance with privacy laws and regulations.

Use Python, ```requests```, ```BeautifulSoup``` and/or ```pandas``` to scrape web data:

## Import Libraries

In [1]:
from pathlib import Path
import requests
import pandas as pd
from bs4 import BeautifulSoup

## Define the Target URL

In [2]:
url = "https://news.google.com/rss/search?q=Tesla%20stock&hl=en-US&gl=US&ceid=US:en"

## Send a Request to the Website

Do not forget to check the response status code

In [3]:
headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(url, headers=headers, timeout=10)

print("Status Code:", response.status_code)

if response.status_code == 200:
    print("Request successful.")
else:
    print("Request failed.")

Status Code: 200
Request successful.


## Parse the HTML Content

Use a library to access the HTMl content

In [4]:
!pip install lxml -q

from bs4 import BeautifulSoup

soup = BeautifulSoup(response.content, "xml")
items = soup.find_all("item")

print("Number of items:", len(items))

Number of items: 100


## Identify the Data to be Scraped

Write a couple of sentence on the data you want to scrape

We are scraping recent Tesla stock-related news from the Google News RSS feed.
The dataset contains structured information including article titles, publication dates, source names, and article links.
This data can be used for further analysis, such as identifying news trends or comparing news activity with Tesla stock price movements over time.

## Extract Data

Find specific elements and extract text or attributes from elements (handle pagination if necessary)

In [5]:
import pandas as pd

articles = []

for item in items:
    title = item.find("title")
    link = item.find("link")
    pub_date = item.find("pubDate")
    source = item.find("source")

    articles.append({
        "title": title.get_text(strip=True) if title else None,
        "link": link.get_text(strip=True) if link else None,
        "published_at": pub_date.get_text(strip=True) if pub_date else None,
        "source": source.get_text(strip=True) if source else None
    })

scraped_df = pd.DataFrame(articles)

print("Number of scraped rows:", len(scraped_df))
scraped_df.head()

Number of scraped rows: 100


,title,link,published_at,source
0,SpaceX and Tesla Shares Are Trading Like Twins...,https://news.google.com/rss/articles/CBMifkFVX...,"Tue, 30 Jun 2026 20:35:00 GMT",Barron's
1,"Tesla Leads Magnificent Seven Recovery, Analys...",https://news.google.com/rss/articles/CBMiogFBV...,"Mon, 29 Jun 2026 20:05:00 GMT",Investor's Business Daily
2,It Took Tesla 10 Years to Perform Its First St...,https://news.google.com/rss/articles/CBMilwFBV...,"Wed, 01 Jul 2026 08:48:00 GMT",Yahoo Finance
3,SpaceX Stock Gets Buy Rating From a Tesla Bull...,https://news.google.com/rss/articles/CBMie0FVX...,"Wed, 01 Jul 2026 08:52:00 GMT",Barron's
4,Tesla Deliveries Should Show a Second Straight...,https://news.google.com/rss/articles/CBMiowFBV...,"Wed, 01 Jul 2026 11:17:00 GMT",Barron's


## Store Data in a Structured Format

Give a brief overview of the data collected (e.g. count, fields, ...)

In [6]:
scraped_df["published_at"] = pd.to_datetime(scraped_df["published_at"], errors="coerce", utc=True)
scraped_df["date"] = scraped_df["published_at"].dt.date

scraped_df = scraped_df.dropna(subset=["title", "published_at"])
scraped_df = scraped_df.drop_duplicates(subset=["title"])
scraped_df = scraped_df.sort_values("published_at", ascending=False)

print("Number of Tesla news articles:", len(scraped_df))
print("Columns:", list(scraped_df.columns))

scraped_df.head()

Number of Tesla news articles: 99
Columns: ['title', 'link', 'published_at', 'source', 'date']


,title,link,published_at,source,date
90,Tesla (TSLA) Stock Hits $420 in July 2026 — AI...,https://news.google.com/rss/articles/CBMiwAFBV...,2026-07-01 11:25:18+00:00,FXLeaders,2026-07-01
4,Tesla Deliveries Should Show a Second Straight...,https://news.google.com/rss/articles/CBMiowFBV...,2026-07-01 11:17:00+00:00,Barron's,2026-07-01
38,"Michael Burry shorts Tesla at $416, betting on...",https://news.google.com/rss/articles/CBMirwFBV...,2026-07-01 11:01:18+00:00,Pluang,2026-07-01
82,Tesla registrations in Spain climb 5.6% in Jun...,https://news.google.com/rss/articles/CBMiqwFBV...,2026-07-01 10:51:55+00:00,Investing.com,2026-07-01
42,Tesla Semi Puts PACCAR Stock And Electric Truc...,https://news.google.com/rss/articles/CBMiygFBV...,2026-07-01 10:42:51+00:00,simplywall.st,2026-07-01


## Save the Data

In [7]:
import os

os.makedirs("../data/raw", exist_ok=True)
os.makedirs("../data/processed", exist_ok=True)

scraped_df.to_csv("../data/raw/tesla_news_raw.csv", index=False)
scraped_df.to_csv("../data/processed/tesla_news_cleaned.csv", index=False)

print("Saved raw data to ../data/raw/tesla_news_raw.csv")
print("Saved cleaned data to ../data/processed/tesla_news_cleaned.csv")
print("Final shape:", scraped_df.shape)

Saved raw data to ../data/raw/tesla_news_raw.csv
Saved cleaned data to ../data/processed/tesla_news_cleaned.csv
Final shape: (99, 5)


## Check if file exists

In [8]:
from pathlib import Path
import pandas as pd

check_path = Path("../data/processed/tesla_news_cleaned.csv")

print("File exists:", check_path.exists())

check_df = pd.read_csv(check_path)

print("Rows:", len(check_df))
display(check_df.head())

File exists: True
Rows: 99


,title,link,published_at,source,date
0,Tesla (TSLA) Stock Hits $420 in July 2026 — AI...,https://news.google.com/rss/articles/CBMiwAFBV...,2026-07-01 11:25:18+00:00,FXLeaders,2026-07-01
1,Tesla Deliveries Should Show a Second Straight...,https://news.google.com/rss/articles/CBMiowFBV...,2026-07-01 11:17:00+00:00,Barron's,2026-07-01
2,"Michael Burry shorts Tesla at $416, betting on...",https://news.google.com/rss/articles/CBMirwFBV...,2026-07-01 11:01:18+00:00,Pluang,2026-07-01
3,Tesla registrations in Spain climb 5.6% in Jun...,https://news.google.com/rss/articles/CBMiqwFBV...,2026-07-01 10:51:55+00:00,Investing.com,2026-07-01
4,Tesla Semi Puts PACCAR Stock And Electric Truc...,https://news.google.com/rss/articles/CBMiygFBV...,2026-07-01 10:42:51+00:00,simplywall.st,2026-07-01


In [9]:
scraped_df.head()
scraped_df.isnull().sum()

title           0
link            0
published_at    0
source          0
date            0
dtype: int64